In [1]:
from src.data import DataPipeline
from src.train import conv_rnn_training
from src.dataset import SARDataset
from src.inference import (
    load_checkpoint,
    predict_full_extent,
    reference_extent,
    masked_mae,
    list_checkpoints,
    best_val_loss,
)
import torch
from torch.utils.data import random_split
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import logging
from pathlib import Path
from skimage.filters import threshold_otsu

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger(__name__)

In [ ]:
# Data dirs
DATA_DIR = "data/preprocessed"
LABEL_PATH = "data/labels/sample_2018_2024_normalized.tif"
WEIGHTS_DIR = Path("checkpoints")
LOGS_DIR = Path("logs")

# Data loader sizes
PATCH_SIZE = 128
BATCH_SIZE = 2

# Hyperparameters
ARCH = ["ConvGRU", "ConvLSTM"]
HIDDEN_CHANNELS = [16, 32]
LEARNING_RATE = [1e-3, 5e-4]
KERNEL_SIZE = [3, 5]

# Epochs
EPOCHS = 50

# Constants
HEAD_CHANNELS = [16]
VAL_FRACTION = 0.2
POS_WEIGHT = 3.26
INPUT_CHANNELS = 2
SEED = 0

In [4]:
def fetch_data(skip_raw=True, skip_preproc=True):
    # Data engineering
    pipeline = DataPipeline()
    if not skip_raw:
        pipeline.raw()
    if not skip_preproc:
        pipeline.preprocessed()
        pipeline.labels()
        
fetch_data()

In [5]:
def train():
    
    # Training
    logger.info("Training procedure started.")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        logger.info(f"Device: {torch.cuda.get_device_name(0)}")
        
    dataset = SARDataset(data_dir=DATA_DIR, label_path=LABEL_PATH, patch_size=PATCH_SIZE)

    n_val = max(1, int(len(dataset) * VAL_FRACTION))
    n_train = len(dataset) - n_val
    generator = torch.Generator().manual_seed(SEED)
    train_set, val_set = random_split(dataset, [n_train, n_val], generator=generator)
       
    WEIGHTS_DIR.mkdir(exist_ok=True)
    LOGS_DIR.mkdir(exist_ok=True)

    for arch in ARCH:
        for hidden in HIDDEN_CHANNELS:
            for kernel in KERNEL_SIZE:
                for lr in LEARNING_RATE:
                    weights_str = f"{arch}_" + f"hidden{hidden}-" + f"kernel{kernel}-" + f"lr{lr}".replace(".", "p")
                    weights_path = WEIGHTS_DIR / f"{weights_str}.pth"
                    history_path = LOGS_DIR / f"{weights_str}.parquet"
                    
                    if weights_path.exists():
                        logger.warning(f"Checkpoint for {weights_str} already exists. Skipping...")
                        continue
                    
                    logger.info(f"Starting ARCH {weights_str}")
                    
                    history = conv_rnn_training(
                        input_channels=INPUT_CHANNELS,
                        kernel_size=kernel,
                        hidden_channels=hidden,
                        head_channels=HEAD_CHANNELS,
                        lrate=lr,
                        train_dataset=train_set,
                        val_dataset=val_set,
                        batch_size=BATCH_SIZE,
                        epochs=EPOCHS,
                        pos_weight=POS_WEIGHT,
                        device=device,
                        weights_path=weights_path,
                        arch=arch,
                        patience=8
                    )
                    
                    history.to_parquet(history_path, index=False)
                
train()

2026-08-27 02:04:51,638 | INFO | __main__ | Training procedure started.
2026-08-27 02:04:51,753 | INFO | __main__ | Device: NVIDIA GeForce RTX 3050 6GB Laptop GPU
2026-08-27 02:04:51,772 | WARNING | __main__ | Checkpoint for ConvGRU_hidden16-kernel3-lr0p001 already exists. Skipping...
2026-08-27 02:04:51,773 | WARNING | __main__ | Checkpoint for ConvGRU_hidden16-kernel3-lr0p0005 already exists. Skipping...
2026-08-27 02:04:51,774 | WARNING | __main__ | Checkpoint for ConvGRU_hidden16-kernel5-lr0p001 already exists. Skipping...
2026-08-27 02:04:51,775 | WARNING | __main__ | Checkpoint for ConvGRU_hidden16-kernel5-lr0p0005 already exists. Skipping...
2026-08-27 02:04:51,777 | WARNING | __main__ | Checkpoint for ConvGRU_hidden32-kernel3-lr0p001 already exists. Skipping...
2026-08-27 02:04:51,777 | WARNING | __main__ | Checkpoint for ConvGRU_hidden32-kernel3-lr0p0005 already exists. Skipping...
2026-08-27 02:04:51,778 | WARNING | __main__ | Checkpoint for ConvGRU_hidden32-kernel5-lr0p001 a

In [6]:
def evaluate(ncols=4):
    
    # Inference
    logger.info("Evaluation started.")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        logger.info(f"Device: {torch.cuda.get_device_name(0)}")
        
    dataset = SARDataset(data_dir=DATA_DIR, label_path=LABEL_PATH, patch_size=PATCH_SIZE)
    label, mask = reference_extent(dataset)
    
    # Best runs first, runs without a log last
    checkpoints = sorted(
        list_checkpoints(WEIGHTS_DIR),
        key=lambda p: (
            best_val_loss(p.stem, LOGS_DIR) is None,
            best_val_loss(p.stem, LOGS_DIR) or 0.0
        )
    )
    
    results = []
    
    for weights_path in checkpoints:
        val = best_val_loss(weights_path.stem, LOGS_DIR)
        
        model, hyperparams = load_checkpoint(weights_path, device=device)
        pred = predict_full_extent(model, dataset, device=device, batch_size=BATCH_SIZE)
        
        # Covers every patch, training ones included, so this flatters the model
        # more than the held-out val_loss does
        mae = masked_mae(pred, label, mask)
        logger.info(f"{weights_path.stem} | val_loss {val:.5f} | full-extent MAE {mae:.5f}")
        
        results.append((weights_path.stem, hyperparams, val, mae, pred))
        
        # The 6GB card does not have room for several of these at once
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
    
    # Label first, then one panel per checkpoint
    nrows = -(-(len(results) + 1) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4.8 * nrows))
    axes = np.atleast_1d(axes).ravel()
    
    axes[0].imshow(label * mask, vmin=0, vmax=1)
    axes[0].set_title("label (ground truth)", fontweight="bold")
    
    for ax, (name, hyperparams, val, mae, pred) in zip(axes[1:], results):
        ax.imshow(pred * mask, vmin=0, vmax=1)
        subtitle = f"val_loss {val:.5f}" if val is not None else "no log"
        ax.set_title(f"{name}\n{subtitle} | MAE {mae:.5f}", fontsize=8)
    
    for ax in axes:
        ax.axis("off")
    
    plt.tight_layout()
    plt.show()
    
    # Architecture comparison is easier to read as a table than off the panels
    summary = pd.DataFrame([
        {
            "run": name,
            "arch": hyperparams["arch"],
            "hidden": hyperparams["hidden_channels"],
            "kernel": hyperparams["kernel_size"],
            "val_loss": val,
            "mae": mae,
        }
        for name, hyperparams, val, mae, _ in results
    ])
    
    display(summary)
    
    if summary["arch"].nunique() > 1:
        display(
            summary
            .groupby("arch")[["val_loss", "mae"]]
            .agg(["min", "mean"])
            .round(5)
        )
    
    return results, summary

results, summary = evaluate()

2026-08-27 02:04:51,811 | INFO | __main__ | Evaluation started.
2026-08-27 02:04:51,813 | INFO | __main__ | Device: NVIDIA GeForce RTX 3050 6GB Laptop GPU


KeyboardInterrupt: 